# 📊 Modelos Tabulares para Predição de RUL — CMAPSS FD001

## Contexto

Com os dados devidamente preparados — sensores normalizados, target com cap de 125 ciclos e rolling features calculadas — este notebook implementa e avalia três modelos de machine learning clássicos para predição de Vida Útil Remanescente (RUL):

- **Regressão Linear** — baseline interpretável, estabelece o limite inferior de performance esperado
- **Random Forest** — ensemble de árvores de decisão, robusto a outliers e não assume linearidade
- **XGBoost** — gradient boosting, estado da arte em dados tabulares e amplamente usado na literatura CMAPSS

Modelos tabulares tratam cada ciclo como uma observação independente — sem memória temporal explícita. As rolling features calculadas no notebook anterior são o que aproxima esses modelos do comportamento sequencial dos dados: em vez de o modelo ver a sequência diretamente, ele recebe estatísticas da janela recente como features estáticas.

## Métricas de avaliação

**RMSE (Root Mean Squared Error)**  
Métrica padrão de regressão — penaliza erros grandes mais do que erros pequenos. É a métrica mais usada para comparação entre papers no CMAPSS.

**S-score (NASA PHM Score)**  
Métrica assimétrica específica da literatura CMAPSS. Penaliza predições tardias (RUL previsto > RUL real) mais severamente do que predições antecipadas — refletindo o contexto operacional: prever que um motor vai durar mais do que realmente dura é muito mais perigoso do que ser conservador.

## Perguntas que este notebook responde

**1. Qual o baseline mínimo de performance para este problema?**  
A regressão linear estabelece o piso — qualquer modelo mais complexo deve superá-la para justificar sua complexidade.

**2. Modelos baseados em árvores conseguem capturar a não-linearidade da degradação?**  
A EDA mostrou que a degradação é exponencial nos últimos ciclos — Random Forest e XGBoost lidam com isso sem precisar de transformações explícitas.

**3. As rolling features realmente agregam valor preditivo?**  
O feature importance do XGBoost vai revelar quais transformações o modelo considerou mais relevantes.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'sans-serif'

# S-score — métrica assimétrica da literatura CMAPSS
# Penaliza predições tardias (d > 0) mais severamente que predições antecipadas (d < 0)
def s_score(y_true, y_pred):
    d = y_pred - y_true
    return np.sum(np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1))

## 1. Carregamento dos Dados

In [ ]:
X_train  = np.load('data/processed/X_train_tab.npy')
y_train  = np.load('data/processed/y_train_tab.npy')
X_test   = np.load('data/processed/X_test_tab.npy')
y_test   = np.load('data/processed/y_test_tab.npy')
unit_ids = np.load('data/processed/test_unit_ids.npy')

# Reconstruindo os nomes das features (12 sensores + 24 rolling)
SENSORS = [
    'sensor_temp_lpc_outlet', 'sensor_temp_hpc_outlet', 'sensor_temp_lpt_outlet',
    'sensor_pressure_ratio', 'sensor_physical_core_speed', 'sensor_static_hpc_outlet',
    'sensor_fuel_flow_ps30', 'sensor_corrected_core_speed', 'sensor_bypass_ratio',
    'sensor_demanded_fan_speed', 'sensor_lpt_coolant_bleed', 'sensor_bpt_ratio'
]
ROLLING = [f'{s}_mean_30' for s in SENSORS] + [f'{s}_std_30' for s in SENSORS]
ALL_FEATURES = SENSORS + ROLLING

print(f'X_train:  {X_train.shape}')
print(f'X_test:   {X_test.shape}')
print(f'Features: {len(ALL_FEATURES)} ({len(SENSORS)} sensores + {len(ROLLING)} rolling)')

## 2. Baseline — Regressão Linear

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_raw = lr.predict(X_test)
y_pred_raw = np.clip(y_pred_raw, 0, 125)

# Média das 5 predições por motor — padrão da literatura CMAPSS
pred_df = pd.DataFrame({'unit_id': unit_ids, 'pred': y_pred_raw})
mean_pred_lr = pred_df.groupby('unit_id')['pred'].mean().values

rmse_lr = np.sqrt(mean_squared_error(y_test, mean_pred_lr))
ss_lr   = s_score(y_test, mean_pred_lr)

print('=== Regressão Linear ===')
print(f'RMSE:    {rmse_lr:.2f}')
print(f'S-score: {ss_lr:.2f}')

## 3. Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=10,
    max_features=0.5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_raw = rf.predict(X_test)
y_pred_raw = np.clip(y_pred_raw, 0, 125)

pred_df = pd.DataFrame({'unit_id': unit_ids, 'pred': y_pred_raw})
mean_pred_rf = pred_df.groupby('unit_id')['pred'].mean().values

rmse_rf = np.sqrt(mean_squared_error(y_test, mean_pred_rf))
ss_rf   = s_score(y_test, mean_pred_rf)

print('=== Random Forest ===')
print(f'RMSE:    {rmse_rf:.2f}')
print(f'S-score: {ss_rf:.2f}')

## 4. XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.05,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

y_pred_raw = xgb_model.predict(X_test)
y_pred_raw = np.clip(y_pred_raw, 0, 125)

pred_df = pd.DataFrame({'unit_id': unit_ids, 'pred': y_pred_raw})
mean_pred_xgb = pred_df.groupby('unit_id')['pred'].mean().values

rmse_xgb = np.sqrt(mean_squared_error(y_test, mean_pred_xgb))
ss_xgb   = s_score(y_test, mean_pred_xgb)

print('=== XGBoost ===')
print(f'RMSE:    {rmse_xgb:.2f}')
print(f'S-score: {ss_xgb:.2f}')

## 5. Comparação dos Modelos

In [ ]:
# Tabela resumo
resultados = pd.DataFrame({
    'Modelo':   ['Regressão Linear', 'Random Forest', 'XGBoost'],
    'RMSE':     [rmse_lr, rmse_rf, rmse_xgb],
    'S-score':  [ss_lr,   ss_rf,   ss_xgb]
}).sort_values('RMSE').reset_index(drop=True)

display(resultados.style
    .background_gradient(subset=['RMSE'],    cmap='RdYlGn_r')
    .background_gradient(subset=['S-score'], cmap='RdYlGn_r')
    .format({'RMSE': '{:.2f}', 'S-score': '{:.2f}'})
)

| Rank | Modelo           |    RMSE ↓ |  S-score ↓ |
| :--: | :--------------- | --------: | ---------: |
|  🥇  | **XGBoost**      | **17.78** | **766.51** |
|  🥈  | Regressão Linear |     19.60 |     906.49 |
|  🥉  | Random Forest    |     19.61 |    1593.98 |


### Observações — Tabela Comparativa

O XGBoost venceu em ambas as métricas — RMSE 17.78 e S-score 766 — confirmando que o gradient boosting captura melhor a não-linearidade da degradação do que os demais modelos.

O resultado mais surpreendente é o empate virtual entre Regressão Linear e Random Forest (RMSE 19.60 vs 19.61). Esperava-se que o Random Forest, sendo um modelo muito mais complexo, superasse o baseline linear com margem. O empate sugere que as rolling features já linearizaram suficientemente o problema — ao entregar a média móvel como feature estática, a regressão linear recebe uma representação suavizada da tendência de degradação e consegue competir com um ensemble de 200 árvores.

O S-score conta uma história diferente: Random Forest (1593) é mais que o dobro da Regressão Linear (906), mesmo com RMSE similar. Isso indica que o Random Forest, apesar de acertar o valor médio, comete mais erros tardios — prevê RUL maior do que o real com mais frequência — o que o S-score penaliza exponencialmente.



In [ ]:
# Predito vs Real
modelos = {
    'Regressão Linear': mean_pred_lr,
    'Random Forest':    mean_pred_rf,
    'XGBoost':          mean_pred_xgb
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (nome, preds) in zip(axes, modelos.items()):
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    ax.scatter(y_test, preds, alpha=0.6, color='steelblue', s=30)
    ax.plot([0, 125], [0, 125], 'r--', linewidth=1.5, label='Ideal')
    ax.set_xlabel('RUL Real')
    ax.set_ylabel('RUL Previsto')
    ax.set_title(f'{nome}\nRMSE = {rmse:.2f}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.3)

fig.suptitle('Predito vs Real — CMAPSS FD001', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pred_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()

![](midia/predito_real.png)

In [ ]:
# Distribuição dos erros
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (nome, preds) in zip(axes, modelos.items()):
    erros = preds - y_test
    ax.hist(erros, bins=25, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(0, color='darkred', linewidth=1.5, linestyle='--')
    ax.axvline(erros.mean(), color='orange', linewidth=1.5,
               linestyle='--', label=f'Média: {erros.mean():.1f}')
    ax.set_title(f'{nome}', fontweight='bold')
    ax.set_xlabel('Erro (Previsto − Real)')
    ax.set_ylabel('Contagem')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.3)

fig.suptitle('Distribuição dos Erros — CMAPSS FD001', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

![](midia/dist_erros.png)

### Observações — Distribuição dos Erros

Os três modelos apresentam distribuições assimétricas com cauda positiva — a maioria dos erros está à direita do zero, indicando que os modelos tendem a **superestimar o RUL** (preveem que o motor vai durar mais do que realmente dura). Esse é o tipo de erro mais perigoso operacionalmente, e é exatamente o que o S-score penaliza mais severamente.

**Regressão Linear (média 6.6):** distribuição mais espalhada e uniforme, sem um pico dominante. Os erros se distribuem de forma mais homogênea entre -40 e +45.

**Random Forest (média 7.6):** pico concentrado próximo de zero, mas com cauda direita longa chegando a +60 — alguns motores com erros muito grandes de superestimação. Explica o S-score elevado (1593) apesar do RMSE similar aos outros.

**XGBoost (média 5.1):** distribuição mais concentrada ao redor de zero, com o pico mais alto e caudas mais curtas. Menor viés sistemático de superestimação — o que se traduz diretamente no melhor S-score (766).

**Próximo passo → Seção 6: Análise do XGBoost**

Com o XGBoost estabelecido como melhor modelo, aprofundamos a análise com Feature Importance e SHAP para entender *quais variáveis* e *como* cada uma contribui para as predições.

## 6. Análise do Melhor Modelo — XGBoost

In [ ]:
importances = xgb_model.feature_importances_
feat_imp = (pd.DataFrame({'Feature': ALL_FEATURES, 'Importance': importances})
              .sort_values('Importance', ascending=False)
              .reset_index(drop=True))

top20 = feat_imp.head(20)

colors = ['darkred' if '_std_' in f else 'steelblue' if '_mean_' in f else 'gray'
          for f in top20['Feature']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top20['Feature'].str.replace('sensor_', ''),
        top20['Importance'], color=colors, alpha=0.8, edgecolor='white')
ax.invert_yaxis()
ax.set_xlabel('Importância (Gain)', fontsize=11)
ax.set_title('Feature Importance — XGBoost\nCMAPSS FD001',
             fontsize=13, fontweight='bold')
ax.grid(True, axis='x', linestyle='--', alpha=0.4)

legend = [
    Patch(color='gray',      label='Sensor original'),
    Patch(color='steelblue', label='Rolling mean (w=30)'),
    Patch(color='darkred',   label='Rolling std (w=30)'),
]
ax.legend(handles=legend, fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features:')
print(feat_imp.head(10).to_string(index=False))

![](midia/features.png)

### Observações — Feature Importance (XGBoost)

**`temp_lpt_outlet_mean_30`** domina com margem absoluta — a média móvel da temperatura de saída da turbina de baixa pressão responde por quase 40% da importância total do modelo. Não é surpresa: esse sensor tinha correlação de -0.679 com o RUL na EDA, alta monotonicidade e prognostibilidade nas métricas PHM, e padrão exponencial claro nos plots de degradação. O modelo confirmou de forma independente o que a análise exploratória já indicava.

**`fuel_flow_ps30_std_30`** em segundo — e aqui está o achado mais interessante. Não é a média do fluxo de combustível que importa, mas sua **instabilidade**. O desvio padrão móvel captura o aumento de ruído mecânico conforme o motor se degrada — exatamente o fenômeno de heterocedasticidade que plotamos na seção de rolling variance da EDA. O modelo aprendeu que um motor errático no fluxo de combustível está mais próximo da falha do que um com fluxo estável, independente do valor absoluto.

**`static_hpc_outlet`** em terceiro como sensor original — o único sem transformação no top 3, e o que tinha maior correlação com RUL (-0.696) na EDA. Sua presença confirma que a normalização Min-Max preservou o sinal original.

De forma geral, as **rolling features dominam o top 10** — 7 das 10 primeiras são médias ou desvios padrão móveis. Isso valida diretamente as decisões de feature engineering: sem essas transformações, o modelo perderia a maior parte da informação preditiva.

**Próximo passo → SHAP**

A feature importance revela *quanto* cada variável foi usada. O SHAP vai revelar *como* — a direção do efeito e o comportamento por amostra individual.

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(
    shap_values,
    X_test,
    feature_names=[f.replace('sensor_', '') for f in ALL_FEATURES],
    max_display=20,
    plot_size=(11, 6),   # largura, altura
    show=False
)

plt.title(
    'SHAP — Impacto das Features no RUL Previsto',
    fontweight='bold',
    fontsize=12
)

plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

### Observações — SHAP

O beeswarm plot revela a direção e magnitude do efeito de cada feature em cada predição individual — informação que a feature importance nativa não fornece.

**`temp_lpt_outlet_mean_30`** confirma e amplia o que a feature importance mostrou. Os pontos vermelhos (temperatura alta) estão concentrados à esquerda, com valores SHAP chegando a -30 — quando a média móvel da temperatura de saída está alta, o modelo reduz drasticamente o RUL previsto. Os pontos azuis (temperatura baixa) ficam ligeiramente à direita, indicando motor saudável. A separação clara entre as cores mostra que essa feature tem efeito consistente e confiável.

**`physical_core_speed`** apresenta o padrão mais extremo do plot — uma cauda de pontos vermelhos chegando a -30, completamente separada da massa principal de pontos azuis próximos de zero. Isso indica que valores altos de velocidade do núcleo em poucos motores específicos dominam completamente a predição — provavelmente os motores nos últimos ciclos de vida com aceleração exponencial dos sensores.

**`bypass_ratio_std_30`** — ao contrário da maioria, aqui os pontos **azuis** ficam à direita. Instabilidade baixa do bypass ratio aumenta o RUL previsto — o motor estável ainda tem vida. É o inverso dos sensores de temperatura, onde o valor alto indica degradação.

**Divergência entre Feature Importance e SHAP:** `fuel_flow_ps30_std_30` era segundo no ranking de importância mas aparece apenas na 14ª posição no SHAP. Isso acontece porque a feature importance mede frequência de uso nos splits — o modelo pode usar uma feature muitas vezes em splits de baixo impacto. O SHAP, por medir contribuição marginal real, revela que o efeito desse sensor no valor final da predição é menor do que a importância nativa sugeria.

De forma geral, o SHAP confirma que o modelo aprendeu a física do problema: sensores térmicos altos e velocidades elevadas indicam degradação, enquanto instabilidade nas pressões sinaliza motor em fim de vida.

---

### Conclusão — Modelos Tabulares

Os três modelos tabulares demonstraram performance sólida no CMAPSS FD001, com o XGBoost alcançando RMSE de 17.78 e S-score de 766 — resultados competitivos com a literatura.

O achado mais relevante foi o impacto do preprocessing: a mesma regressão linear que com normalização incorreta produzia RMSE de 58, com o preprocessing correto (scaler do treino + média das últimas 5 predições por motor) caiu para 19.60 — redução de 66% sem nenhuma mudança no modelo. Isso ilustra que em problemas de séries temporais industriais, a qualidade do preprocessing frequentemente supera a escolha do algoritmo.

A análise de feature importance e SHAP confirmou os achados da EDA e das métricas PHM: sensores térmicos — especialmente `temp_lpt_outlet` — dominam a predição, e as rolling features agregaram valor real ao capturar tanto a tendência quanto a instabilidade do sinal de degradação.

**Próximo notebook → LSTM**

Os modelos tabulares tratam cada ciclo como observação independente, dependendo das rolling features para capturar contexto temporal. O LSTM vai um passo além — aprende diretamente da sequência de 30 ciclos, sem precisar de features estáticas. A comparação final entre as duas abordagens revelará se a memória temporal explícita do LSTM justifica sua maior complexidade.